In [ ]:
from cubes.construct.buildingconfig import load_building_config
from cubes.construct.building import Building
from cubes.package import envconfig
from cubes.construct.core import materials_evaluator, windows_evaluator
from stable_baselines3 import SAC
from datetime import datetime
from cubes.cubesgym.utils.wrappers import LoggerWrapperCubes
from cubes.cubesgym.utils.callbacks import LoggerEvalCallback


from stable_baselines3.common.callbacks import CallbackList
from stable_baselines3.common.vec_env import DummyVecEnv


bc = load_building_config("input.json")
ec = envconfig.EnvConfig(observe_zone_temperature=True,
                         observe_electricity_demand=True,
                         observe_outside_temperature=True,
                         observe_zone_occupancy=True,
                         observe_zone_co2=True,
                         observe_grid_carbon_intensity=True,
                         control_thermostat_setpoints=True,
                         control_battery_charging=True,
                         observe_outside_temperature_in_x_hours_forecast=[1,24])
building=Building(bc, materials_evaluator(), windows_evaluator())
building.build()
idf = building.get_idf()

In [ ]:
from cubes.package.core import register_environment
environment = "test_env-v1"

register_environment(environment,idf,bc,ec)

In [3]:
episodes = 10
experiment_date = datetime.today().strftime('%Y-%m-%d %H:%M')

# register run name
name = F"SAC-{environment}-episodes_{episodes}({experiment_date})"

In [ ]:
import gym 
env = gym.make(environment)
env = LoggerWrapperCubes(env)
model = SAC('MlpPolicy', env, verbose=1,
    tensorboard_log="./"+environment+"_tensorboard/")

In [ ]:
n_timesteps_episode = env.simulator._eplus_one_epi_len / \
                      env.simulator._eplus_run_stepsize

print(env.simulator._eplus_one_epi_len,env.simulator._eplus_run_stepsize)

In [6]:
env_vec = DummyVecEnv([lambda: env])

In [7]:
callbacks = []

# Set up Evaluation and saving best model
eval_callback = LoggerEvalCallback(
    env_vec,
    best_model_save_path='best_model/' + name + '/',
    log_path='best_model/' + name + '/',
    eval_freq=n_timesteps_episode * 2,
    deterministic=True,
    render=False,
    n_eval_episodes=2)
callbacks.append(eval_callback)

callback = CallbackList(callbacks)

In [8]:
timesteps = episodes * n_timesteps_episode

In [9]:
model.save(env.simulator._env_working_dir_parent + '/' + name)

In [ ]:
model.learn(
    total_timesteps=timesteps,
    log_interval=1,
    callback=callback)

In [11]:
env.close()

[2023-06-20 10:55:54,024] EPLUS_ENV_test_env-v1_MainThread_ROOT INFO:EnergyPlus simulation closed successfully. 
